In [1]:
# =================================================================
# SOTA ISLES-2022: DynUNet (nnU-Net Architecture) Training Engine
# - Hardcoded for DynUNet with Deep Residual Blocks
# - Uses deterministic 70/15/15 train/val/test split (seed=42)
# - FORCES 100 epochs (NO EARLY STOPPING, Kaggle 12h Safe)
# - Includes SwinUNETR in the model factory for future testing
# =================================================================

!pip install -q monai nibabel scikit-learn einops

import os
import logging
import warnings
import sys
import torch
import numpy as np
import nibabel as nib
import nibabel.processing
from collections import defaultdict

# Suppress Kaggle C++ and TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ['CUDA_MODULE_LOADING'] = 'LAZY' 
logging.getLogger('absl').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split 
from tqdm.auto import tqdm

import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

# Importing SOTA architectures
from monai.networks.nets import DynUNet, SwinUNETR, SegResNet
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, NormalizeIntensityd, RandSpatialCropd,
    RandFlipd, RandRotate90d, CastToTyped, EnsureTyped, SpatialPadd
)
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch

# --- 1. KAGGLE PATHS & CONFIGURATION ---
CONFIG = {
    "SEARCH_ROOT": "/kaggle/input/datasets/prosenjitmondol/a-stroke-lesion-segmentation-dataset/ISLES-2022",
    "SAVE_DIR": "/kaggle/working/",
    
    # Training the nnU-Net clone with Residual blocks
    "model_name": "DynUNet", 
    
    "roi_size": (64, 64, 64),
    "batch_size": 1,
    "epochs": 100,      # Safe for 12-hour GPU limit
    "lr": 1e-4,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "seed": 42,
    
    "split": {"train": 0.70, "val": 0.15, "test": 0.15}
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
print(f"🚀 Initializing {CONFIG['model_name']} Engine | Training for {CONFIG['epochs']} epochs on {CONFIG['device']}")

# --- 2. DATA PROCESSING ---
def prepare_isles_data(root):
    subjects = defaultdict(dict)
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(('.nii', '.nii.gz')):
                full_path = os.path.join(dirpath, f)
                sub_id = next((p for p in full_path.split(os.sep) if 'sub-' in p.lower()), os.path.basename(dirpath))
                f_l = f.lower()
                if 'dwi' in f_l: subjects[sub_id]['dwi'] = full_path
                elif 'adc' in f_l: subjects[sub_id]['adc'] = full_path
                elif 'flair' in f_l: subjects[sub_id]['flair'] = full_path
                elif any(x in f_l for x in ['msk', 'mask', 'lesion']): subjects[sub_id]['msk'] = full_path
    
    data = [f for s, f in subjects.items() if all(k in f for k in ['dwi', 'adc', 'flair', 'msk'])]
    data = sorted(data, key=lambda x: list(x.values())[0])  
    return data

class ISLESDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data, self.transform = data, transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        p = self.data[idx]
        dwi = nib.load(p['dwi'])
        adc_r = nib.processing.resample_from_to(nib.load(p['adc']), dwi, order=1)
        flr_r = nib.processing.resample_from_to(nib.load(p['flair']), dwi, order=1)
        msk_r = nib.processing.resample_from_to(nib.load(p['msk']), dwi, order=0)

        img = np.stack([np.nan_to_num(dwi.get_fdata()), np.nan_to_num(adc_r.get_fdata()), np.nan_to_num(flr_r.get_fdata())], 0)
        lbl = np.expand_dims(np.nan_to_num(msk_r.get_fdata()), 0)

        del dwi, adc_r, flr_r, msk_r
        d = {"image": img.astype(np.float32), "label": lbl.astype(np.float32)}
        return self.transform(d) if self.transform else d

# --- 3. AUGMENTATIONS ---
xforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    SpatialPadd(keys=["image", "label"], spatial_size=CONFIG["roi_size"]),
    RandSpatialCropd(keys=["image", "label"], roi_size=CONFIG["roi_size"], random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    CastToTyped(keys=["image", "label"], dtype=[torch.float32, torch.float32]),
    EnsureTyped(keys=["image", "label"]),
])

test_transforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CastToTyped(keys=["image"], dtype=[torch.float32]),
    EnsureTyped(keys=["image"]),
])

# --- 4. ADVANCED MODEL FACTORY ---
def get_model(model_name):
    if model_name == "DynUNet":
        # MONAI's implementation of the legendary nnU-Net
        return DynUNet(
            spatial_dims=3, 
            in_channels=3, 
            out_channels=1,
            kernel_size=[[3,3,3], [3,3,3], [3,3,3], [3,3,3], [3,3,3]], 
            strides=[[1,1,1], [2,2,2], [2,2,2], [2,2,2], [2,2,2]],
            upsample_kernel_size=[[2,2,2], [2,2,2], [2,2,2], [2,2,2]], 
            filters=[16, 32, 64, 128, 256],
            dropout=0.1,
            res_block=True  # CRITICAL: Adds residual connections to boost score past 82%
        ).to(CONFIG["device"])
        
    elif model_name == "SwinUNETR":
        # Vision Transformer approach for global stroke context
        return SwinUNETR(
            img_size=CONFIG["roi_size"],
            in_channels=3,
            out_channels=1,
            feature_size=24, 
            use_checkpoint=True 
        ).to(CONFIG["device"])
        
    else:
        raise ValueError(f"Unknown model: {model_name}")

# --- 5. TRAIN / VAL / TEST SPLIT (70/15/15) ---
def split_data(data, seed=42):
    train_ratio = CONFIG["split"]["train"]
    val_ratio = CONFIG["split"]["val"]
    test_ratio = CONFIG["split"]["test"]
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Splits must sum to 1.0"

    train_data, temp_data = train_test_split(data, train_size=train_ratio, random_state=seed, shuffle=True)
    temp_size = len(temp_data)
    if temp_size == 0: return train_data, [], []
    
    val_size = int(round(val_ratio / (val_ratio + test_ratio) * temp_size))
    return train_data, temp_data[:val_size], temp_data[val_size:]

# --- 6. TRAINING ENGINE ---
def run():
    torch.manual_seed(CONFIG["seed"])
    np.random.seed(CONFIG["seed"])
    data = prepare_isles_data(CONFIG["SEARCH_ROOT"])
    if len(data) == 0:
        print("❌ No complete subjects found in SEARCH_ROOT.")
        return

    train_data, val_data, test_data = split_data(data, seed=CONFIG["seed"])
    print(f"Dataset sizes -> Total: {len(data)} | Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    t_ldr = DataLoader(ISLESDataset(train_data, xforms), batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
    v_ldr = DataLoader(ISLESDataset(val_data, test_transforms), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
    test_ldr = DataLoader(ISLESDataset(test_data, test_transforms), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

    loss_fn = DiceFocalLoss(include_background=False, sigmoid=True, squared_pred=True, gamma=2.0)
    metric = DiceMetric(include_background=False, reduction="mean")
    
    m = get_model(CONFIG["model_name"])
    opt = optim.AdamW(m.parameters(), lr=CONFIG["lr"], weight_decay=1e-5)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["epochs"])
    scaler = GradScaler('cuda') if torch.cuda.is_available() else None

    best_val = 0.0
    best_model_path = os.path.join(CONFIG["SAVE_DIR"], f"{CONFIG['model_name']}_best_val.pth")

    for ep in range(CONFIG["epochs"]):
        print(f"\nEpoch {ep+1:03d}/{CONFIG['epochs']}")
        m.train()
        l_sum, train_steps = 0.0, 0
        
        for b in tqdm(t_ldr, desc="Train", leave=False):
            img, msk = b["image"].to(CONFIG["device"]), b["label"].to(CONFIG["device"])
            opt.zero_grad()
            
            if scaler:
                with autocast('cuda'):
                    out = m(img)
                    loss = loss_fn(out, msk)
                scaler.scale(loss).backward()
                
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=2.0)
                
                scaler.step(opt)
                scaler.update()
            else:
                out = m(img)
                loss = loss_fn(out, msk)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=2.0)
                opt.step()
                
            l_sum += loss.item()
            train_steps += 1

        avg_loss = l_sum / train_steps if train_steps > 0 else 0.0
        sch.step()

        # Validation
        m.eval()
        metric.reset()
        with torch.no_grad():
            for vb in tqdm(v_ldr, desc="Val", leave=False):
                vi, vm = vb["image"].to(CONFIG["device"]), vb["label"].to(CONFIG["device"])
                vo = sliding_window_inference(vi, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(vo)]
                metric(y_pred=preds, y=vm)
                del vi, vm, vo

        cur_val = metric.aggregate().item() if len(val_data) > 0 else 0.0
        metric.reset()
        print(f"Loss: {avg_loss:.4f} | Val Dice: {cur_val:.4f}")

        if cur_val > best_val:
            best_val = cur_val
            torch.save(m.state_dict(), best_model_path)
            print(f"🌟 New best validation Dice: {best_val:.4f} -> saved to {best_model_path}")
        else:
            print(f"Current best remains: {best_val:.4f} (Forced run, continuing...)")

        if CONFIG["device"].type == "cuda":
            torch.cuda.empty_cache()

    # Final Evaluation
    if os.path.exists(best_model_path):
        print(f"\n🔁 Loading best model from {best_model_path} for final evaluation.")
        m.load_state_dict(torch.load(best_model_path, map_location=CONFIG["device"]))

    if len(val_data) > 0:
        m.eval()
        metric.reset()
        with torch.no_grad():
            for vb in tqdm(v_ldr, desc="Final Val Eval", leave=False):
                vi, vm = vb["image"].to(CONFIG["device"]), vb["label"].to(CONFIG["device"])
                vo = sliding_window_inference(vi, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(vo)]
                metric(y_pred=preds, y=vm)
                del vi, vm, vo
        print(f"\n✅ Final Validation Dice (F1): {metric.aggregate().item():.4f}")

    if len(test_data) > 0:
        m.eval()
        metric.reset()
        with torch.no_grad():
            for tb in tqdm(test_ldr, desc="Test Eval", leave=False):
                ti, tm = tb["image"].to(CONFIG["device"]), tb["label"].to(CONFIG["device"])
                to = sliding_window_inference(ti, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(to)]
                metric(y_pred=preds, y=tm)
                del ti, tm, to
        print(f"\n🎯 Test Dice (F1): {metric.aggregate().item():.4f}")

if __name__ == "__main__":
    run()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.4 MB/s eta 0:00:0000:0100:01


E0000 00:00:1773776706.484785      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773776706.550006      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773776706.995104      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773776706.995144      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773776706.995147      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773776706.995150      55 computation_placer.cc:177] computation placer already registered. Please check linka

🚀 Initializing DynUNet Engine | Training for 100 epochs on cuda
Dataset sizes -> Total: 250 | Train: 175 | Val: 38 | Test: 37

Epoch 001/100


Train:   0%|          | 0/175 [00:00<?, ?it/s]

Val:   0%|          | 0/38 [00:00<?, ?it/s]

...

Loss: 0.8609 | Val Dice: 0.6447
Current best remains: 0.6658 (Forced run, continuing...)

🔁 Loading best model from /kaggle/working/DynUNet_best_val.pth for final evaluation.


Final Val Eval:   0%|          | 0/38 [00:00<?, ?it/s]


✅ Final Validation Dice (F1): 0.6658


Test Eval:   0%|          | 0/37 [00:00<?, ?it/s]


🎯 Test Dice (F1): 0.5661
